# Exercises (Student) - MCP Client with LLM

In [ ]:
!pip install -q mcp nest_asyncio requests

In [ ]:

import os
from pathlib import Path
MCP_HTTP_TOKEN = os.getenv("MCP_HTTP_TOKEN", "devtoken123")
USE_REAL_LLM = False  # flip True if GITHUB_TOKEN is set


In [ ]:
import asyncio
import nest_asyncio
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

nest_asyncio.apply()


In [ ]:
%%writefile server.py
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("DemoServer")

@mcp.tool()
def add(a: int, b: int) -> int:
    "Add two numbers."
    return a + b

@mcp.tool()
def multiply(a: int, b: int) -> int:
    "Multiply two numbers."
    return a * b

@mcp.tool()
def greet(name: str) -> str:
    "Return a greeting string."
    return f"Hello, {name}!"

if __name__ == "__main__":
    mcp.run()


## Exercise 1 (provide answer)

## Exercise 1 (provide answer)

**Pourquoi STDIO est plus simple que HTTP pour le developpement MCP local :**

- **Aucune infrastructure reseau** : pas de port a choisir, pas de pare-feu, pas de configuration TLS/CORS - le client lance simplement le serveur comme sous-processus et communique via stdin/stdout.
- **Cycle de vie simplifie** : le serveur demarre et s'arrete avec le client (meme processus parent), donc pas de serveur "orphelin" qui continue de tourner en arriere-plan apres les tests.
- **Pas d'authentification a gerer** : contrairement a HTTP ou il faut typiquement un jeton/en-tete d'autorisation, STDIO est deja isole au niveau du systeme d'exploitation (seul le processus parent qui a spawn le sous-processus peut lui parler).
- **Debogage plus direct** : les erreurs du serveur peuvent etre observees directement sur stderr (`errlog=True`), sans avoir besoin d'outils comme curl/Postman ou de logs serveur separes.

En contrepartie, STDIO ne convient qu'a un usage local (un seul client, un seul serveur co-localises) : HTTP reste necessaire pour exposer un serveur MCP a plusieurs clients distants ou sur un reseau.

## Exercise 2

In [ ]:

import asyncio
import nest_asyncio
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

nest_asyncio.apply()

async def ex2_connect():
    params = StdioServerParameters(command="python", args=["server.py"])
    async with stdio_client(params, errlog=True) as (r, w):
        async with ClientSession(r, w) as session:
            await session.initialize()


In [ ]:
# In a new cell
await ex2_connect()
print("Exercise 2: OK (connected and initialized)")


Exercise 2: OK (connected and initialized)


## Exercise 3

In [ ]:
async def ex3_list():
    params = StdioServerParameters(command="python", args=["server.py"])
    async with stdio_client(params, errlog=True) as (r, w):
        async with ClientSession(r, w) as session:
            await session.initialize()
            resources = await session.list_resources()
            print("RESOURCES:", resources)
            tools = await session.list_tools()
            for t in tools.tools:
                print(t.name, t.inputSchema.get("properties", {}))


In [ ]:
await ex3_list()

RESOURCES: meta=None nextCursor=None resources=[]
add {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


## Exercise 4

## Exercise 4

**Comment la conversion en outil LLM se produit :**

Chaque outil MCP expose deja des metadonnees structurees via le protocole - un `name`, une `description` (tiree de la docstring Python de la fonction decoree par `@mcp.tool()`), et un `inputSchema` au format JSON Schema, genere automatiquement par FastMCP a partir des annotations de type de la fonction (par exemple `a: int, b: int` devient `{"a": {"type": "integer"}, "b": {"type": "integer"}}`).

`convert_to_llm_tool` se contente donc de reformater ces memes informations dans le format attendu par l'API de function-calling d'un LLM (`{"type": "function", "function": {"name": ..., "description": ..., "parameters": {...}}}`). Aucune nouvelle information n'est inventee : le schema JSON du serveur MCP devient directement le schema de parametres de l'outil cote LLM, ce qui garantit que les arguments generes par le modele correspondent exactement a la signature Python reelle de l'outil.

In [ ]:

def convert_to_llm_tool(tool):
    return {
        "type": "function",
        "function": {
            "name": tool.name,
            "description": tool.description or "mcp tool",
            "parameters": {
                "type": "object",
                "properties": tool.inputSchema.get("properties", {}),
                "required": tool.inputSchema.get("required", []),
            },
        },
    }


## Exercise 5

**Plan & execute:** Use stub (or real) LLM to propose `tool_calls`, then execute them and print results for a prompt like “Add 2 to 20.”

In [ ]:

# NOTE: `call_llm` (next cell) calls `stub_plan(...)` when USE_REAL_LLM is False,
# but the original notebook never defines it. Here is a minimal keyword + number
# extraction planner that infers which registered tool to call from the prompt.
import re

def stub_plan(prompt: str, functions):
    """Very small rule-based planner: pick a tool by keyword, and use the
    numbers found in the prompt as its 'a' and 'b' arguments."""
    numbers = [int(n) for n in re.findall(r"-?\d+", prompt)]
    available = {f["function"]["name"] for f in functions}
    prompt_lower = prompt.lower()

    if len(numbers) < 2:
        return []

    if "multiply" in available and any(k in prompt_lower for k in ["multiply", "times", "product"]):
        name = "multiply"
    elif "add" in available and any(k in prompt_lower for k in ["add", "plus", "sum"]):
        name = "add"
    else:
        return []

    a, b = numbers[0], numbers[1]
    return [{"name": name, "args": {"a": a, "b": b}}]


In [ ]:

import asyncio
import json
import nest_asyncio
from typing import Any, Dict, List
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

nest_asyncio.apply()

def call_llm(prompt: str, functions: List[Dict[str, Any]], use_real: bool = False):
    if not use_real:
        return stub_plan(prompt, functions)
    token = os.getenv("GITHUB_TOKEN")
    if not token:
        raise RuntimeError("Set GITHUB_TOKEN or use stub planner.")
    from azure.ai.inference import ChatCompletionsClient
    from azure.core.credentials import AzureKeyCredential
    client = ChatCompletionsClient("https://models.inference.ai.azure.com", AzureKeyCredential(token))
    resp = client.complete(
        model="gpt-4o",
        messages=[{"role": "system", "content": "Plan MCP tool calls."},{"role": "user", "content": prompt}],
        tools=functions,
        temperature=0,
        max_tokens=400,
    )
    calls = []
    msg = resp.choices[0].message
    for tc in msg.tool_calls or []:
        args = tc.function.arguments
        args_json = json.loads(args) if isinstance(args, str) else args
        calls.append({"name": tc.function.name, "args": args_json})
    return calls


In [ ]:
async def ex5_run(prompt: str = "Add 2 to 20"):
    params = StdioServerParameters(command="python", args=["server.py"])
    async with stdio_client(params, errlog=True) as (r, w):
        async with ClientSession(r, w) as session:
            await session.initialize()
            tools = await session.list_tools()
            functions = [convert_to_llm_tool(t) for t in tools.tools]
            calls = call_llm(prompt, functions, use_real=USE_REAL_LLM)
            print("tool_calls:", calls)
            for call in calls:
                result = await session.call_tool(call["name"], arguments=call["args"])
                print("result:", [getattr(c, "text", str(c)) for c in result.content])


In [ ]:
await ex5_run("Add 2 to 20")

tool_calls: [{'name': 'add', 'args': {'a': 2, 'b': 20}}]
result: ['22']


## Optional - add multiply(a, b) and rerun

In [ ]:

# server.py already registers `multiply` as a tool (see the completed server.py cell above).
# stub_plan() also recognizes multiply-style prompts, so this should route to the multiply tool.
await ex5_run("Multiply 6 by 7")
